# 07 - Basic multi-objective optimization

This notebook introduces multi-objective optimization with explicit objective outputs, ACCIM parameters, and NSGA-II execution.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `OptimisationSimulation`
- `OptimisationSimulation.set_parameters`
- `OptimisationSimulation.set_problem`
- `OptimisationSimulation.run_optimisation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `optim.discover_available_outputs`
- `optim.select_outputs`
- `optim.apply_outputs_preflight`
- `optim.set_output_readers`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Basic optimization: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

`PROCESSES` is the number of simulations run in parallel: half of the logical CPUs by default.

In [1]:
import os
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import OptimisationSimulation

pd.set_option("display.max_columns", 160)

# Simulations run in parallel: half of the logical CPUs.
PROCESSES = max(1, (os.cpu_count() or 2) // 2)


## Basic optimization: set project paths and discover IDF/EPW files

Here we declare the baseline project assets (IDFs, EPWs, and output folder).

For Basic optimization, this is the configuration anchor. If asset names change, update this block before touching execution cells.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_list`, `epw_list`, `RESULTS_DIRNAME`.

In [2]:
PROJECT_ROOT = Path.cwd()

idf_list = [
	"OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
	"OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_low_performance.idf",
]
epw_list = ["Seville.epw", "Sydney.epw"]
RESULTS_DIRNAME = "nb07_basic_optimization_results"

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


IDF files found: [True, True]
EPW files found: [True, True]


## Basic optimization: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [3]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


Baseline file count: 83


## Basic optimization: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [4]:
building = ef.get_building(idf_list[0])


## Basic optimization: apply runtime-reduction settings

Apply runtime reductions to keep notebook runs faster during exploration.

This block shortens exploratory runs by reducing simulation burden. Use it for prototyping and switch to fuller settings when producing final results.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [5]:
accim.utils.reduce_runtime(
    idf_object=building,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"


Solar distribution has been set to MinimalShadowing.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 15000 days, and it has been modified to 200 days.
Number of Timesteps per Hour was previously set to 6 days, and it has been modified to 2 days.


## Basic optimization: initialize optimization simulation object

Create the optimization simulation object with objective context.

This creates the optimization simulation object that will manage evaluators, objectives, and algorithms. The next cells configure objectives and decision variables before solving.

Key arguments used in this cell and why they matter:
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `optim`.

In [6]:
optim = OptimisationSimulation(
	building=building,
	parameters_type="accim custom model",
	epws=epw_list,
	output_freqs=["hourly"],
)



--------------------------------------------------------
Adaptive-Comfort-Control-Implemented Model (ACCIM) v0.7.8
--------------------------------------------------------

This tool allows to apply adaptive setpoint temperatures. 
For further information, please read the documentation: 
https://accim.readthedocs.io/en/master/
For a visual understanding of the tool, please visit the following jupyter notebooks:
-    Using addAccis() to apply adaptive setpoint temperatures
https://accim.readthedocs.io/en/master/jupyter_notebooks/addAccis/using_addAccis.html
-    Using rename_epw_files() to rename the EPWs for proper data analysis after simulation
https://accim.readthedocs.io/en/master/jupyter_notebooks/rename_epw_files/using_rename_epw_files.html
-    Using runEp() to directly run simulations with EnergyPlus
https://accim.readthedocs.io/en/master/jupyter_notebooks/runEp/using_runEp.html
-    Using the class Table() for data analysis
https://accim.readthedocs.io/en/master/jupyter_notebo

Air distribution effectiveness (always 1) Schedule has been added
VRF Heating Cooling (Northern Hemisphere) Schedule has been added
DefaultFanEffRatioCurve Curve:Cubic Object has been added
VRFTUCoolCapFT Curve:Cubic Object has been added
VRFTUHeatCapFT Curve:Cubic Object has been added
VRFCoolCapFTBoundary Curve:Cubic Object has been added
VRFCoolEIRFTBoundary Curve:Cubic Object has been added
CoolingEIRLowPLR Curve:Cubic Object has been added
VRFHeatCapFTBoundary Curve:Cubic Object has been added
VRFHeatEIRFTBoundary Curve:Cubic Object has been added
HeatingEIRLowPLR Curve:Cubic Object has been added
DefaultFanPowerRatioCurve Curve:Exponent Object has been added
DXHtgCoilDefrostEIRFT Curve:Biquadratic Object has been added
VRFCoolCapFT Curve:Biquadratic Object has been added
VRFCoolCapFTHi Curve:Biquadratic Object has been added
VRFCoolEIRFT Curve:Biquadratic Object has been added
VRFCoolEIRFTHi Curve:Biquadratic Object has been added
VRFHeatCapFT Curve:Biquadratic Object has been ad

AHST_Sch_Floor_1_Residential_Living_Occupants Schedule has been added
ACST_Sch_Floor_1_Residential_Living_Occupants Schedule has been added
AHST_Sch_Floor_2_Residential_Living_Occupants Schedule has been added
ACST_Sch_Floor_2_Residential_Living_Occupants Schedule has been added
Added - SetComfTemp Program
Added - CountHours_Floor_1_Residential_Living_Occupants Program
Added - CountHours_Floor_2_Residential_Living_Occupants Program
Added - SetAppLimits Program
Added - ApplyCAT Program
Added - SetAST_Master Program with 109 lines


Added - 46 per-ComfStand SetAST subprograms
Added - SetAST Program (base program that calls SetAST_Master)
Added - SetASTnoTol Program
Added - CountHoursNoApp_Floor_1_Residential_Living_Occupants Program
Added - SetGeoVarFloor_1_Residential_Living_Occupants Program
Added - CountHoursNoApp_Floor_2_Residential_Living_Occupants Program
Added - SetGeoVarFloor_2_Residential_Living_Occupants Program
Added - SetInputData Program
Added - SetVOFinputData Program
Added - SetVST Program
Added - ApplyAST_Floor_1_Residential_Living_Occupants Program
Added - ApplyAST_Floor_2_Residential_Living_Occupants Program
Added - Comfort Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature Output Variable
Added - Adaptive Heating Setpoint Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature_No Tolerance Output Variable
Added - Adaptive Heating Setpoint Temperature_No Tolerance Output Variable
Added - z_test_ComfStand Output Variable
Added - z_test_ACSTaul Output Variab

## Basic optimization: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [7]:

available_outputs = optim.discover_available_outputs(
    reduce_sim_time=True,
    prefer="testsimeplus",
)
df_meters_available = available_outputs["meters"]
df_vars_available = available_outputs["variables"]
outputs_meta = available_outputs["meta"]
print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


Solar distribution is already set to MinimalShadowing, therefore no action has been performed.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 200 days, and it has been modified to 200 days.
Number of Timesteps per Hour was previously set to 2 days, and it has been modified to 2 days.


Discovery source: testsimeplus
Meters available: 6
Variables available: 144


## Basic optimization: validate and apply output configuration

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `df_output_meter`: final meter table used by the simulation.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `_`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [8]:

objective_meters = ["Heating:Electricity", "Cooling:Electricity"]
selection = optim.select_outputs(
    meters=objective_meters,
    match="case_insensitive",
    on_missing="raise",
    suggest=True,
)
df_meters_sel = selection["meters"]

optim.apply_outputs_preflight(
    df_meters_sel=df_meters_sel,
    df_vars_sel=None,
    clean_mode="all",
    validate_before_apply=False,
    validate_after_apply=True,
    on_missing="raise",
)
optim.set_output_readers(df_output_meter=df_meters_sel)


## Basic optimization: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.
- `minimize_outputs`: boolean vector indicating which objectives are minimized.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`.

In [9]:
accis_parameters = {
	"CustAST_m": (0.15, 0.95),
	"CustAST_n": (8, 23),
	"CustAST_ASToffset": (1.0, 6.0),
	"CustAST_ASTall": (8, 16),
	"CustAST_ASTaul": (28, 38),
}

optim.set_parameters(accis_params_dict=accis_parameters)
optim.set_problem(minimize_outputs=[True, True])


## Basic optimization: estimate expected simulation workload

Before execution, we estimate workload size from plan cardinality and solver budget settings.

Use this checkpoint to align runtime expectations with available compute resources before launching.

NSGA-II evaluates the initial population and then one generation of offspring after another until it reaches `evaluations`. It creates the offspring in pairs, so with an odd population size each generation would evaluate one individual more than the estimate; an even population size keeps the run equal to it (here 2 generations x 4 individuals = 8 simulations per EPW).

Key arguments used in this cell and why they matter:
- `evaluations`: total number of optimization evaluations.
- `population_size`: population size for the evolutionary algorithm.
- `epws`: list of EPW weather files used to run scenarios.

Variables created in this cell:
- `EVALUATIONS`: reused in subsequent workflow blocks.
- `POPULATION_SIZE`: reused in subsequent workflow blocks.
- `estimated`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [10]:
EVALUATIONS = 8
POPULATION_SIZE = 4

estimated = optim.estimate_optimisation_sims(
	evaluations=EVALUATIONS,
	population_size=POPULATION_SIZE,
	epws=epw_list,
)
print("Estimated simulations:", estimated)


Estimated simulations
  evaluations    : 8
  population_size: 4
  EPWs           : 2 (Seville.epw, Sydney.epw)
  sims per EPW   : 8  (2 generation(s) × 4)
  TOTAL          : 16
Estimated simulations: 16


## Basic optimization: run NSGA-II optimization and inspect Pareto flags

This cell runs the optimization algorithm with the current objectives and search settings.

Treat the returned dataframe as the optimization trace and confirm Pareto-related columns before decision-support steps.

Key arguments used in this cell and why they matter:
- `epws`: list of EPW weather files used to run scenarios.
- `out_dir`: output folder for generated artifacts and results.
- `evaluations`: total number of optimization evaluations.
- `population_size`: population size for the evolutionary algorithm.
- `algorithm`: optimization algorithm to execute.
- `processes`: number of parallel processes for simulation runs.
- `keep_df`: policy for storing output dataframes.
- `keep_sim_files`: policy for preserving simulation files.
- `keep_sim_files_batch_size`: batch size used when preserving simulation files.

Variables created in this cell:
- `outputs_optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check objective columns and the `pareto-optimal` indicator before post-analysis.
- Confirm the selected algorithm and evaluation budget were applied as intended.

In [11]:

outputs_optim = optim.run_optimisation(
    epws=epw_list,
    out_dir=RESULTS_DIRNAME,
    evaluations=EVALUATIONS,
    population_size=POPULATION_SIZE,
    algorithm="NSGAII",
    processes=PROCESSES,
    keep_df="all",
    keep_sim_files="non-dominated",
    keep_sim_files_batch_size=POPULATION_SIZE,
)
print("Rows:", len(outputs_optim))
if "pareto-optimal" in outputs_optim.columns:
    print(outputs_optim["pareto-optimal"].value_counts(dropna=False))
print(outputs_optim.head())


[run_optimisation] Starting optimisation (NSGAII): 2 case(s) (2 to run, 0 resumed) | ~8 sims/case (2 generation(s) x 4, evaluations=8) | ~16 sims in total | 6 process(es)


[run_optimisation] Case 1/2 started (idf=OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance, epw=Seville); ~8 simulation(s) expected.


[run_optimisation] Case 1/2 | generation 1: evaluating 4 individual(s)...


[run_optimisation] Case 1/2 | gen 1 sim 1/4 | case 1/~8 | total 1/~16 (6.2%) | elapsed 0:00:18 | ETA ~0:04:37


[run_optimisation] Case 1/2 | gen 1 sim 2/4 | case 2/~8 | total 2/~16 (12.5%) | elapsed 0:00:19 | ETA ~0:02:12


...

[run_optimisation] Case 2/2 | gen 2 sim 2/4 | case 6/~8 | total 14/~16 (87.5%) | elapsed 0:00:46 | ETA ~0:00:07


[run_optimisation] Case 2/2 | gen 2 sim 3/4 | case 7/~8 | total 15/~16 (93.8%) | elapsed 0:00:46 | ETA ~0:00:03


[run_optimisation] Case 2/2 | gen 2 sim 4/4 | case 8/~8 | total 16/~16 (100.0%) | elapsed 0:00:46 | ETA ~0:00:00


[run_optimisation] Case 2/2 completed (idf=OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance, epw=Sydney): 8 simulation(s), 2 generation(s) in 0:00:18.


  [info] simulation_summary updated for optimisation after run_optimisation.
Rows: 16
pareto-optimal
True     10
False     6
Name: count, dtype: int64
   CustAST_m  CustAST_n  CustAST_ASToffset  CustAST_ASTall  CustAST_ASTaul  \
0   0.598539  11.003156           5.677525       15.341989       28.092900   
1   0.453973   8.465261           1.613604       12.837707       32.384504   
2   0.727201  16.171459           2.505375       11.897146       32.211305   
3   0.412520  11.284147           5.496093       15.637320       28.918965   
4   0.450841  20.210192           5.496093       15.633339       32.361396   

   Heating:Electricity  Cooling:Electricity  \
0         6.433156e+09         4.026680e+10   
1         4.261673e+09         1.112820e+11   
2         4.179179e+10         2.534195e+10   
3         2.755030e+09         6.168981e+10   
4         2.943618e+10         1.926803e+10   

                                simulation_directory  \
0  nb07_basic_optimization_results\BESOS_

### Reading the result

Inspect objective columns and `pareto-optimal` status before moving to compromise or robustness analysis.

## Basic optimization: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [12]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)


Generated files detected: 164
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusmtr.csv
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.audit
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.bnd
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.csv
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.eio
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.end
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.err
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.eso
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mdd
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mtd
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mtr
 - nb07_bas

## Basic optimization: preview cleanup actions with dry-run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [13]:
# Force-remove every generated artifact inside the dedicated result folder,
# including generated IDF files.
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


Files planned for deletion: 164
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusmtr.csv
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.audit
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.bnd
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.csv
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.eio
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.end
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.err
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.eso
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mdd
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mtd
 - nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mtr
 - nb07_b

['nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusmtr.csv',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.audit',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.bnd',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.csv',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.eio',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.end',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.err',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.eso',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mdd',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mtd',
 'nb07_basic_optimization_results/BESOS_Output_1504/4VJP1ERXASOENGDRM2BZ/eplusout.mtr',
 'nb07_basic_optimization_resu

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

`DELETE_RESULTS` is False by default, so nothing is deleted: notebooks 09 and 10 read these results. Set it to True to delete the files listed above.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [14]:
DELETE_RESULTS = False

if DELETE_RESULTS:
    artifact_cleaner.delete_generated_files(
        allow_patterns=[f"{RESULTS_DIRNAME}/**"],
        deny_patterns=[],
        dry_run=False,
        remove_empty_dirs=True,
    )
